# Cadence — local Whisper server for development

Runs `faster-whisper` on Colab's free GPU and exposes it over a public tunnel, so caption generation during development doesn't spend Groq's shared quota. See `colab/README.md` in the repo for the full picture — this notebook is just the server.

**Runtime**: make sure a GPU is attached before running — `Runtime` menu → `Change runtime type` → `T4 GPU`.

Run the three cells below in order. The last cell prints a URL — copy it into `COLAB_WHISPER_URL` in your `.env.local` and restart your dev server.

In [ ]:
# Cell 1 — install dependencies and the cloudflared tunnel binary.
!pip install -q fastapi uvicorn python-multipart nest-asyncio faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# Cell 2 — load the model and define the server.
#
# "medium" is the sweet spot here: roughly twice as fast as large-v3 on a
# free-tier T4 and easily good enough for captions, which matters because the
# tunnel gives up on any request that takes more than ~100 seconds. Move to
# "large-v3" if you're specifically evaluating transcript quality, or "small"
# if you want the fastest possible turnaround while working on something else.
#
# SHARED_SECRET is optional. Leave it blank to accept any request — the tunnel
# URL itself is an unguessable random string, which is a reasonable bar for a
# throwaway dev session. Set it here and mirror it in COLAB_SHARED_SECRET in
# .env.local for an actual check.

import os, tempfile, threading
from fastapi import FastAPI, UploadFile, Header, HTTPException
from faster_whisper import WhisperModel

MODEL_SIZE = "medium"
SHARED_SECRET = ""

print(f"Loading {MODEL_SIZE} — this can take a minute or two the first time...")
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
print("Model loaded.")

app = FastAPI()

# Only one transcription may touch the GPU at a time — two concurrent runs on a
# free-tier T4 is a straightforward way to run out of VRAM. Requests queue here
# rather than failing, and because the endpoint runs off the event loop (see
# below) queueing costs nothing else.
_gpu = threading.Lock()


def _check_auth(authorization: str | None):
    if SHARED_SECRET and authorization != f"Bearer {SHARED_SECRET}":
        raise HTTPException(status_code=401, detail="Missing or wrong bearer token.")


@app.get("/health")
def health():
    # Deliberately unauthenticated: a health check that itself needs the
    # secret can't tell "down" apart from "wrong token" from the caller's
    # side, and the app's fallback logic only needs to know "is anything here".
    return {"ok": True, "model": MODEL_SIZE}


# This mirrors the OpenAI-compatible request shape the app already sends to
# Groq (multipart file + response_format=verbose_json + word-level
# timestamps), so nothing on the Next.js side needs to know it's talking to a
# different provider — see lib/ai/transcribe.ts's transcribeBlob.
#
# Deliberately a plain `def`, not `async def`: FastAPI runs sync handlers in a
# threadpool, whereas an async one executes on the event loop itself. Since
# model.transcribe() blocks, an async handler would freeze the whole server
# for the duration — including /health, which the app polls to decide whether
# this server is alive. It would answer "down" mid-job and the app would give
# up on a transcription that was running perfectly well.
@app.post("/audio/transcriptions")
def transcribe(
    file: UploadFile,
    authorization: str | None = Header(None),
):
    _check_auth(authorization)

    suffix = os.path.splitext(file.filename or "")[1] or ".audio"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        # Sync read, to match the sync handler.
        tmp.write(file.file.read())
        path = tmp.name

    try:
        with _gpu:
            segments_gen, _info = model.transcribe(
                path, word_timestamps=True, vad_filter=True
            )

            segments, words, text_parts = [], [], []
            # faster-whisper yields lazily, so the work actually happens here —
            # inside the lock, which is where it needs to stay.
            for seg in segments_gen:
                text = seg.text.strip()
                if not text:
                    continue
                segments.append({"start": seg.start, "end": seg.end, "text": text})
                text_parts.append(text)
                for w in seg.words or []:
                    words.append({"word": w.word, "start": w.start, "end": w.end})

        print(f"Transcribed a chunk: {len(segments)} segments.")
        return {"text": " ".join(text_parts), "segments": segments, "words": words}
    finally:
        os.remove(path)

In [ ]:
# Cell 3 — start the server and the tunnel, then print the public URL.
#
# Keep this cell running: stopping it (or the notebook idling out) ends the
# tunnel, and the app falls back to Groq automatically the next time it tries.

import nest_asyncio, uvicorn, subprocess, threading, time, re

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for the tunnel to come up...\n")
for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print(f"Whisper server is live at: {url}")
        print(f"\nSet in .env.local:\n  COLAB_WHISPER_URL={url}")
        break